# Prueba de recuperación Top-K con PubMed

Este notebook prueba una consulta externa contra PubMed hasta obtener un ranking Top-K de artículos.

El flujo implementado es:

```text
Consulta del usuario
  → generación de términos para PubMed
  → PubMed ESearch: recuperación de PMIDs
  → PubMed EFetch: título, resumen, autores y DOI
  → BM25 y SBERT opcional
  → ranking Top-K de evidencias
```

La prueba no incluye RAG, generación de respuestas, clasificación de veracidad ni ajuste con Triplet Loss. La API se usa únicamente para comprobar la recuperación de documentos externos.

## Alcance y advertencias

- El texto original del usuario se conserva en español.
- Para este ejemplo se formula una expresión de búsqueda en inglés porque la mayoría de los registros recuperados por PubMed están indexados en inglés.
- La formulación de la consulta es determinista y pedagógica; no representa todavía un traductor o generador de consultas con LLM.
- La similitud o puntuación de recuperación indica relación textual, no demuestra que una afirmación sea verdadera.
- El resultado sirve como prueba de integración de PubMed y no reemplaza el benchmark de Seminario 1 con datos y juicios de relevancia controlados.

In [ ]:
from collections import Counter
import json
import math
import re
import unicodedata
import xml.etree.ElementTree as ET
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from IPython.display import display

USER_AFFIRMATION = "¿Las vacunas causan malestar estomacal?"
TOP_K = 5
RETMAX = 20
RUN_SBERT = False
SBERT_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
NCBI_TOOL = "recomendador-fuentes-seminario1"
NCBI_EMAIL = ""  # Opcional: colocar un correo propio si se usa la API con frecuencia.

print(f"Afirmación del usuario: {USER_AFFIRMATION}")
print(f"Top-K solicitado: {TOP_K}")

In [ ]:
STOPWORDS = {
    "a", "al", "causan", "con", "de", "del", "el",
    "en", "la", "las", "los", "malestar", "que",
    "un", "una", "y",
}

def normalize_text(text):
    """Normaliza texto para tokenización básica y comparación léxica."""
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9\s-]", " ", text.lower())

def tokenize(text):
    return [
        token
        for token in normalize_text(text).split()
        if token not in STOPWORDS and len(token) > 1
    ]

def generate_pubmed_query(affirmation):
    """Genera una consulta reproducible para el ejemplo de vacunas."""
    normalized = normalize_text(affirmation)
    if "vacuna" in normalized and ("estomacal" in normalized or "malestar" in normalized):
        return (
            "vaccines[Title/Abstract] AND ("
            "gastrointestinal[Title/Abstract] OR nausea[Title/Abstract] "
            "OR \"abdominal pain\"[Title/Abstract]"
            ")"
        )

    terms = [token for token in tokenize(affirmation) if len(token) > 2]
    return " AND ".join(f"{term}[Title/Abstract]" for term in terms)

PUBMED_QUERY = generate_pubmed_query(USER_AFFIRMATION)
# Consulta equivalente, sin etiquetas de campo, para puntuar localmente con BM25.
RANKING_QUERY = "vaccines gastrointestinal adverse effects nausea abdominal pain"

print("Consulta enviada a PubMed:")
print(PUBMED_QUERY)
print("Consulta usada para el ranking local:")
print(RANKING_QUERY)

## 1. PubMed ESearch: recuperar identificadores

La primera llamada consulta la base PubMed y devuelve una lista ordenada de PMIDs. Se usa `retmode=json`, `retmax` y orden por relevancia.

In [ ]:
NCBI_BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def ncbi_request(endpoint, params, response_type="json"):
    request_params = dict(params)
    request_params["tool"] = NCBI_TOOL
    if NCBI_EMAIL:
        request_params["email"] = NCBI_EMAIL

    url = f"{NCBI_BASE_URL}/{endpoint}?{urlencode(request_params)}"
    request = Request(
        url,
        headers={"User-Agent": f"{NCBI_TOOL}/0.1"},
    )

    try:
        with urlopen(request, timeout=30) as response:
            payload = response.read()
    except HTTPError as error:
        raise RuntimeError(f"PubMed respondió HTTP {error.code}: {error.reason}") from error
    except URLError as error:
        raise RuntimeError(f"No se pudo conectar con PubMed: {error.reason}") from error

    if response_type == "json":
        return json.loads(payload.decode("utf-8"))
    return payload

def pubmed_search(term, retmax=20):
    response = ncbi_request(
        "esearch.fcgi",
        {
            "db": "pubmed",
            "term": term,
            "retmax": retmax,
            "retmode": "json",
            "sort": "relevance",
        },
    )
    result = response["esearchresult"]
    return {
        "count": int(result.get("count", 0)),
        "pmids": result.get("idlist", []),
    }

search_result = pubmed_search(PUBMED_QUERY, retmax=RETMAX)
print(f"Resultados disponibles en PubMed: {search_result['count']:,}")
print(f"PMIDs recuperados: {len(search_result['pmids'])}")
print(search_result["pmids"][:10])

assert isinstance(search_result["pmids"], list)
assert len(search_result["pmids"]) > 0, "La consulta no devolvió PMIDs; revisa los términos."

## 2. PubMed EFetch: recuperar el contenido de cada artículo

Con los PMIDs se solicita el XML de los registros para extraer los campos que alimentarán el ranking: título, resumen, autores, revista, año y DOI.

In [ ]:
def element_text(element):
    if element is None:
        return ""
    return " ".join("".join(element.itertext()).split())

def extract_year(article):
    for path in (
        ".//ArticleDate/Year",
        ".//JournalIssue/PubDate/Year",
    ):
        year = article.findtext(path)
        if year:
            return year

    medline_date = article.findtext(".//JournalIssue/PubDate/MedlineDate")
    return medline_date or ""

def extract_authors(article):
    authors = []
    for author in article.findall(".//AuthorList/Author"):
        collective = author.findtext("CollectiveName")
        if collective:
            authors.append(collective)
            continue

        last_name = author.findtext("LastName") or ""
        fore_name = author.findtext("ForeName") or ""
        name = " ".join(part for part in (fore_name, last_name) if part)
        if name:
            authors.append(name)
    return "; ".join(authors)

def extract_doi(article):
    for identifier in article.findall(".//PubmedData/ArticleIdList/ArticleId"):
        if identifier.attrib.get("IdType") == "doi":
            return element_text(identifier)
    return ""

def parse_pubmed_article(article):
    pmid = article.findtext(".//MedlineCitation/PMID") or ""
    title = element_text(article.find(".//ArticleTitle"))
    abstract_parts = [
        element_text(node)
        for node in article.findall(".//Abstract/AbstractText")
        if element_text(node)
    ]
    abstract = " ".join(abstract_parts)
    journal = article.findtext(".//Journal/Title") or ""
    text = " ".join(part for part in (title, abstract) if part)

    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "authors": extract_authors(article),
        "journal": journal,
        "year": extract_year(article),
        "doi": extract_doi(article),
        "text": text,
        "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
    }

def pubmed_fetch(pmids):
    payload = ncbi_request(
        "efetch.fcgi",
        {
            "db": "pubmed",
            "id": ",".join(pmids),
            "retmode": "xml",
        },
        response_type="xml",
    )
    root = ET.fromstring(payload)
    return [parse_pubmed_article(article) for article in root.findall(".//PubmedArticle")]

articles = pubmed_fetch(search_result["pmids"])
documents = pd.DataFrame(articles)

print(f"Artículos estructurados: {len(documents)}")
display(documents[["pmid", "title", "journal", "year", "doi"]].head(5))

assert len(documents) > 0
assert {"pmid", "title", "abstract", "authors", "doi"}.issubset(documents.columns)

## 3. Ranking local con BM25

BM25 puntúa la relación léxica entre la consulta y el texto combinado de título y resumen. El resultado conserva el PMID y el enlace de PubMed para trazabilidad.

In [ ]:
def bm25_rank(query_text, document_frame, top_k=5, k1=1.5, b=0.75):
    collection_tokens = [tokenize(text) for text in document_frame["text"]]
    query_tokens = tokenize(query_text)
    document_count = len(collection_tokens)
    average_length = np.mean([len(tokens) for tokens in collection_tokens]) or 1.0

    document_frequency = Counter(
        token
        for tokens in collection_tokens
        for token in set(tokens)
    )
    scores = []

    for tokens in collection_tokens:
        term_counts = Counter(tokens)
        document_length = len(tokens)
        score = 0.0

        for term in query_tokens:
            frequency = term_counts[term]
            if frequency == 0:
                continue

            df = document_frequency[term]
            idf = math.log(1 + (document_count - df + 0.5) / (df + 0.5))
            normalization = k1 * (
                1 - b + b * document_length / average_length
            )
            score += idf * (
                frequency * (k1 + 1)
                / (frequency + normalization)
            )
        scores.append(score)

    ranked = document_frame.copy()
    ranked["bm25_score"] = scores
    ranked = ranked.sort_values("bm25_score", ascending=False).reset_index(drop=True)
    ranked.insert(0, "rank", np.arange(1, len(ranked) + 1))
    return ranked.head(top_k)

bm25_top_k = bm25_rank(RANKING_QUERY, documents, top_k=TOP_K)

display(
    bm25_top_k[
        ["rank", "pmid", "title", "authors", "year", "doi", "bm25_score", "pubmed_url"]
    ]
)

In [ ]:
assert 0 < len(bm25_top_k) <= TOP_K
assert bm25_top_k["rank"].tolist() == list(range(1, len(bm25_top_k) + 1))
assert bm25_top_k["bm25_score"].notna().all()
assert bm25_top_k["bm25_score"].is_monotonic_decreasing
assert bm25_top_k["pmid"].astype(str).str.len().gt(0).all()

print("Validación Top-K BM25: OK")
print("La salida contiene artículos reales de PubMed con PMID y enlace trazable.")

## 4. Ranking opcional con embeddings SBERT

Esta celda es opcional. Si `RUN_SBERT = True`, representa la consulta y los artículos mediante embeddings y los ordena por similitud coseno. Si la dependencia no está instalada, el notebook conserva el resultado BM25.

In [ ]:
sbert_top_k = None

if not RUN_SBERT:
    print("SBERT omitido. Cambia RUN_SBERT = True para comparar embeddings.")
else:
    try:
        from sentence_transformers import SentenceTransformer

        sbert_model = SentenceTransformer(SBERT_MODEL_NAME)
        query_embedding = sbert_model.encode(
            [RANKING_QUERY],
            normalize_embeddings=True,
        )[0]
        document_embeddings = sbert_model.encode(
            documents["text"].tolist(),
            normalize_embeddings=True,
        )
        sbert_scores = np.dot(document_embeddings, query_embedding)
        sbert_ranked = documents.copy()
        sbert_ranked["cosine_similarity"] = sbert_scores
        sbert_ranked = sbert_ranked.sort_values("cosine_similarity", ascending=False).reset_index(drop=True)
        sbert_ranked.insert(0, "rank", np.arange(1, len(sbert_ranked) + 1))
        sbert_top_k = sbert_ranked.head(TOP_K)
        display(
            sbert_top_k[
                ["rank", "pmid", "title", "year", "doi", "cosine_similarity", "pubmed_url"]
            ]
        )
    except ImportError:
        print("SBERT no disponible. Instala el extra con: .venv/bin/python -m pip install -e '.[sbert]'")
    except Exception as error:
        print(f"SBERT no pudo ejecutarse; BM25 sigue disponible. Detalle: {type(error).__name__}: {error}")

## Resultado de la prueba

El notebook comprueba la parte de recuperación externa hasta Top-K. El siguiente paso metodológico sería validar la relevancia de los artículos recuperados y decidir si esta fuente formará parte del corpus controlado de la tesis. La generación de una respuesta fundamentada pertenece a una etapa posterior.